# 第八章：大语言模型 (LLM) 与对齐技术

## 从 Transformer 到 RLHF (Reinforcement Learning from Human Feedback)——理解 ChatGPT 背后的技术栈

2022 年底 ChatGPT 横空出世，改变了 AI 发展轨迹。本章从 Transformer 架构出发，
介绍预训练、监督微调 (SFT (Supervised Fine-Tuning)+LoRA实战)、RLHF/DPO (Direct Preference Optimization) 对齐技术、Prompt Engineering 等核心主题，
让你理解现代 LLM 的工作原理和训练范式。


## 8.1 Transformer：用注意力替代循环

### 为什么需要替代 RNN

RNN 按顺序处理序列——必须算完第 $t-1$ 步才能算第 $t$ 步。这个串行依赖有两个后果：无法并行（100 个 token 的序列需要 100 步顺序计算），长距离依赖受限（梯度跨越多步后衰减）。

Transformer 的核心思想：让序列中的**任意两个位置直接交互**，不需要通过中间步骤传递信息。

### Self-Attention：每个词看所有其他词

输入是一个长度为 $n$ 的 token 序列，每个 token 表示为一个 $d$ 维向量。对每个 token，Self-Attention 做三件事：

1. 产生三个向量：Query $\mathbf{q}$（我要找什么）、Key $\mathbf{k}$（我有什么）、Value $\mathbf{v}$（我的内容）
2. 用我的 Query 去匹配所有 token 的 Key，得到相关性分数
3. 按相关性分数加权聚合所有 token 的 Value

写成矩阵形式：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

其中 $Q, K, V$ 都是 $n \times d$ 的矩阵，分别由输入 $X$ 通过三个可学习的线性投影得到：$Q = XW_Q, K = XW_K, V = XW_V$。

**$QK^T$ 在算什么：** $QK^T$ 是一个 $n \times n$ 的矩阵，位置 $(i, j)$ 的值是第 $i$ 个 token 的 Query 与第 $j$ 个 token 的 Key 的内积。内积越大 → 第 $i$ 个 token 认为第 $j$ 个 token 与它更相关。除以 $\sqrt{d_k}$ 是为了防止点积过大导致 softmax 进入梯度饱和区。

**softmax 的作用：** 将每一行的相关性分数归一化为概率分布（和为 1）——每个 token 对所有其他 token 的注意力权重。然后乘以 $V$，按注意力权重加权聚合所有 token 的 Value。

**复杂度：** $QK^T$ 需要 $n \times d_k \times n = O(n^2 d)$ 次乘法。序列长度翻倍，计算量翻四倍——这是 Transformer 的根本瓶颈，也是位置编码、稀疏注意力等后续研究试图解决的问题。

### 多头注意力

一组 $Q,K,V$ 捕获一种"相关性模式"——比如句法关系。使用 $h$ 组独立的 $Q,K,V$（$h$ 个头），每组在 $d/h$ 维子空间中计算注意力，最后拼接：

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

不同头可以学习不同类型的依赖：主谓关系、指代关系、长距离语义关联等。总计算量与单头相同（$h \times n^2 \times d/h = n^2 d$），但表达能力更强。


### 8.1.1 为什么 Attention 有效：信息检索的视角

Attention 机制可以理解为一个**可微分的键值检索系统**。

**类比：图书馆查资料**

你去图书馆找关于"梯度下降"的书——
- **Query (Q)**：你的查询词"梯度下降"
- **Key (K)**：每本书的索引标签
- **Value (V)**：每本书的实际内容

系统计算 Query 与每个 Key 的**相似度**（注意力权重），然后用这些权重对 Value 做**加权求和**——最相关的书贡献最多内容。

在 Transformer 中，Q、K、V 都来自同一条输入序列（Self-Attention），所以每个 token 都在"查询"序列中所有其他 token：

$$

\text{Output}_i = \sum_{j=1}^{n} \underbrace{\text{softmax}\left(\frac{q_i^T k_j}{\sqrt{d_k}}\right)}_{\text{token i 对 token j 的关注度}} \cdot v_j

$$

#### 为什么除以 $\sqrt{d_k}$？

当 $d_k$ 很大时（如 64 或 128），点积 $q_i^T k_j$ 的值会很大——因为它是 $d_k$ 个随机变量的和，方差约为 $d_k$。

大点积值导致 softmax 输出趋向 one-hot（只有最大的那个接近 1，其余接近 0）→ 梯度趋近于 0 → 训练不动。

除以 $\sqrt{d_k}$ 将方差缩放回 1，保持 softmax 输出的平滑性，让梯度能正常流动。

#### Self-Attention vs Cross-Attention

| 类型 | Q 来源 | K,V 来源 | 用途 |
|------|--------|---------|------|
| **Self-Attention** | 当前序列 | 当前序列 | 捕捉序列内部关系（GPT 的基础） |
| **Cross-Attention** | 当前序列 | 另一序列 | 翻译（源语言 → 目标语言）、多模态 |

GPT 只使用因果 Self-Attention，而原始 Transformer 的 Decoder 同时使用两种。


#### 自注意力的计算复杂度分析

自注意力的核心计算为 $\text{softmax}(QK^T / \sqrt{d_k})V$。逐步骤分析复杂度（$n$ = 序列长度，$d$ = 模型维度）：

**1. $QK^T$：** $Q \in \mathbb{R}^{n \times d}$，$K^T \in \mathbb{R}^{d \times n}$。矩阵乘法需要 $n \times d \times n = \mathcal{O}(n^2 d)$ 次乘法。结果是一个 $n \times n$ 的注意力权重矩阵——每对 token 之间都有一个注意力分数。

**2. Softmax + 除以 $\sqrt{d_k}$：** 对每行做 softmax，复杂度 $\mathcal{O}(n^2)$（每行的 $n$ 个元素都参与指数和求和）。

**3. 乘以 $V$：** 注意力矩阵 $n \times n$ 乘以 $V \in \mathbb{R}^{n \times d}$，复杂度 $\mathcal{O}(n^2 d)$。

**总复杂度：$\mathcal{O}(n^2 d)$**——由 $n^2$ 主导。这是 Transformer 的根本瓶颈：当序列长度翻倍时，计算量翻四倍。处理 2048 token 需要 4M 次注意力计算；处理 8192 token 需要 64M 次——增长是二次方而非线性。

多头注意力中每个头的 $d_k = d / h$，总计算量不变（$h \times n^2 \times (d/h) = n^2 d$），但允许模型在不同子空间中并行计算注意力。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# === 从零实现缩放点积注意力 ===
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, n_heads, seq_len, d_k)
    """
    d_k = Q.size(-1)
    
    # scores: (batch, n_heads, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # 转置两个维度
    
    # 可选的因果掩码（decoder 防止看到未来 token）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights  # 返回结果

# 演示：一个小序列的自注意力
batch, n_heads, seq_len, d_k = 1, 2, 5, 4
Q = torch.randn(batch, n_heads, seq_len, d_k)  # 标准正态分布随机张量
K = torch.randn(batch, n_heads, seq_len, d_k)  # 标准正态分布随机张量
V = torch.randn(batch, n_heads, seq_len, d_k)  # 标准正态分布随机张量

output, attn = scaled_dot_product_attention(Q, K, V)
print(f"输出 shape: {output.shape}")   # (1, 2, 5, 4)
print(f"注意力矩阵 shape: {attn.shape}")  # (1, 2, 5, 5)
print(f"注意力权重 (head 0):\n{attn[0, 0].detach().numpy().round(3)}")  # 从计算图分离
print(f"每行和: {attn[0, 0].sum(-1).detach().numpy()}")  # 应该都是 1.0


### 位置编码 (Positional Encoding)

Transformer 没有循环结构，无法感知 token 的**顺序**。位置编码为每个位置注入唯一的位置信息。

原始论文使用三角函数位置编码：

$$

PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)

$$

$$

PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)

$$

现代 LLM 多使用**可学习位置编码**（如 GPT）或 **RoPE (Rotary Position Embedding)**。


In [ ]:
# === Sinusoidal Positional Encoding ===
import torch
import math
def sinusoidal_positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)  # 全零张量
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)  # 增加一个维度
    div_term = torch.exp(torch.arange(0, d_model, 2).float() *  # 整数序列张量
                         (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # 返回结果

pe = sinusoidal_positional_encoding(50, 64)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))  # 创建画布
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')  # 显示图像
plt.colorbar(label='Encoding Value')  # 颜色条
plt.xlabel('Embedding Dimension'); plt.ylabel('Position')  # 设置x轴标签
plt.title('Sinusoidal Positional Encoding (50 positions, 64 dims)')  # 设置图表标题
plt.savefig('../fig/positional_encoding.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 8.2 GPT 系列：从 GPT-1 到 GPT-4

| 模型 | 年份 | 参数量 | 核心创新 |
|------|------|--------|---------|
| **GPT-1** | 2018 | 117M | 无监督预训练 + 有监督微调 |
| **GPT-2** | 2019 | 1.5B | Zero-shot 能力，认为足够大的语言模型可直接完成任务 |
| **GPT-3** | 2020 | 175B | In-context learning, few-shot prompting |
| **GPT-4** | 2023 | ~1.8T (估计) | 多模态，更强的推理能力 |
| **ChatGPT** | 2022 | GPT-3.5 + RLHF (Reinforcement Learning from Human Feedback) | 指令遵循 + 对话能力 |

### 预训练：Next Token Prediction

GPT 的训练目标极其简单——**预测下一个 token**：

$$

\mathcal{L}_{\text{pretrain}} = -\sum_{t=1}^{T} \log P(x_t | x_{<t})

$$

给定前文 $x_{<t}$，模型需要预测 $x_t$。这个看似简单的任务，
在足够大的模型和海量数据上，迫使模型学会语法、事实知识、推理能力。

### GPT 的核心：因果自注意力 (Causal Self-Attention)

与原始 Transformer 不同，GPT 使用**因果掩码 (Causal Mask)**，确保每个 token 只能看到**它之前**的 token（不能偷看未来）。


In [ ]:
# === 因果掩码可视化 ===
import matplotlib.pyplot as plt
import torch
seq_len = 8
mask = torch.tril(torch.ones(seq_len, seq_len))  # 下三角矩阵

fig, ax = plt.subplots(figsize=(5, 5))  # 创建子图网格
ax.imshow(mask, cmap='Blues')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, '✓' if mask[i, j] else '✗', ha='center', va='center',
                color='green' if mask[i, j] else 'red', fontsize=10)
ax.set_xlabel('Key Position (attended from)')
ax.set_ylabel('Query Position (can attend to)')
ax.set_title('Causal Attention Mask\nToken t can only attend to tokens ≤ t')
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/causal_mask.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 8.3 监督微调 (Supervised Fine-Tuning, SFT)

预训练的 LLM 是一个"文字接龙"机器——它擅长续写，但**不擅长遵循指令**。
SFT 通过在（指令，回答）数据集上继续训练，让模型学会"听懂人话"。

### SFT 流程

1. 收集高质量（指令，回答）对（人工标注或蒸馏）
2. 将指令格式化为固定模板（如 ChatML）
3. 在预训练模型上继续训练（通常 2-5 epochs）
4. **仅计算答案部分的损失**，指令部分不参与 loss 计算

### 损失掩码 (Loss Masking)

```
输入:  <|user|>解释一下什么是反向传播<|assistant|>反向传播是一种...
损失:  0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1
                ↑ 仅在这些位置计算损失
```

### 核心思想

SFT 的本质是**行为克隆 (Behavioral Cloning)**——让模型模仿人类给出的高质量回答。
简单但有效：一个经过良好 SFT 的 7B 模型可以媲美未对齐的 70B 模型。


In [ ]:
# === SFT 训练数据格式示例 ===
# 使用 HuggingFace datasets 加载真实 SFT 数据集
# 注意：需要安装: pip install datasets

# 模拟 SFT 数据格式
import torch
sft_example = {
    "messages": [
        {"role": "user", "content": "用 Python 实现快速排序"},
        {"role": "assistant", "content": "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)"}
    ]
}

# 格式化函数
def format_chatml(messages):
    """ChatML 格式"""
    formatted = ""
    for msg in messages:
        formatted += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    formatted += "<|im_start|>assistant\n"  # 提示模型开始生成
    return formatted  # 返回结果

print("ChatML 格式输出:")
print(format_chatml(sft_example["messages"]))

# loss mask 生成
def create_loss_mask(token_ids, assistant_start_positions):
    """
    仅在 assistant 部分的 token 上计算损失
    assistant_start_positions: 每个 assistant 回复的起始位置列表
    """
    mask = torch.zeros(len(token_ids))  # 全零张量
    for start in assistant_start_positions:
        mask[start:] = 1  # 简化：从第一个 assistant token 开始
    return mask  # 返回结果

print(f"\nToken IDs: {list(range(20))}")
print(f"Loss Mask: {create_loss_mask(torch.arange(20), [10]).int().tolist()}")  # 整数序列张量


### 8.3.1 SFT (Supervised Fine-Tuning) 实战：使用 TRL SFTTrainer

**TRL (Transformer Reinforcement Learning)** 是 HuggingFace 生态中的对齐训练库，提供了 、、 等高层 API。

SFT 的核心挑战不是在模型架构上——而是**数据准备**和**训练配置**。以下展示完整流程：

1. 加载预训练模型与 tokenizer
2. 准备 ChatML 格式的指令数据
3. 配置 LoRA (Low-Rank Adaptation) 高效微调
4. 使用 SFTTrainer 训练

> **关键设计决策**：
> - **Loss Masking**：SFTTrainer 自动对 prompt 部分（user/assistant 前的 system prompt 和 user 消息）mask 掉 loss，只在 assistant 回复上计算损失——这与 8.3 节手写的 loss mask 推导一致
> - **LoRA (Low-Rank Adaptation) vs 全参数微调**：LoRA 可训练参数仅为原模型的 0.1%~1%，在消费级 GPU 上即可完成。全参数微调需要 A100 级别硬件
> - **数据质量 >> 数据量**：精心筛选的 1K 条高质量指令数据，效果优于 10K 条噪声数据


In [ ]:
# === SFT 实战：Qwen2.5-0.5B 指令微调 ===
# 使用 TRL SFTTrainer + LoRA，在 ~10GB 显存上即可运行

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

# 1. 准备数据：ChatML 格式
data = [
    {"messages": [
        {"role": "system", "content": "你是一个有帮助的助手。"},
        {"role": "user", "content": "什么是反向传播？"},
        {"role": "assistant", "content": "反向传播是通过链式法则计算神经网络中每个参数梯度的算法。"}
    ]},
    {"messages": [
        {"role": "user", "content": "用 Python 写一个斐波那契函数"},
        {"role": "assistant", "content": "def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a"}
    ]},
]
# 实际使用时应加载数千条以上的高质量指令数据

# 2. 加载模型与 tokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # 小模型，适合演示
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Qwen 无 pad token，用 eos 替代

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

# 3. LoRA 配置：只训练低秩适配器，冻结原模型
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,               # LoRA rank
    lora_alpha=32,     # 缩放因子
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 目标模块
)

# 4. SFTTrainer：自动处理 ChatML 格式化、loss masking、梯度累积
trainer = SFTTrainer(
    model=model,
    args=TrainingArguments(
        output_dir="./qwen-sft-output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=3,
        logging_steps=10,
        save_strategy="epoch",
        bf16=True,                      # 混合精度
    ),
    train_dataset=Dataset.from_list(data),
    tokenizer=tokenizer,
    peft_config=lora_config,
    max_seq_length=512,
)

# 5. 开始训练
trainer.train()

# 6. 保存 LoRA 适配器（仅几 MB，不是完整模型）
trainer.save_model("./qwen-sft-lora")
print("SFT 完成！LoRA 权重已保存。")


## 8.4 RLHF 与 DPO：让模型符合人类偏好

SFT (Supervised Fine-Tuning) 让模型"会回答"，但回答的质量、安全性、有用性仍可能不佳。
RLHF 和 DPO 旨在让模型的输出**符合人类偏好** (Human Alignment)。

### RLHF 三阶段

```
阶段1: SFT → 阶段2: 奖励模型训练 → 阶段3: PPO 强化学习
```

**阶段 2：奖励模型 (Reward Model, RM)**

- 对同一个 prompt，SFT 模型生成多个回答
- 人类标注员对这些回答**排序**（A > B > C > D）
- 训练一个模型，输入 (prompt, response)，输出**标量奖励** $r$
- 损失函数 (Bradley-Terry 模型)：
  $$\mathcal{L}_{\text{RM}} = -\log \sigma(r_\text{chosen} - r_\text{rejected})$$

**阶段 3：PPO (Proximal Policy Optimization)**

- 使用阶段 2 的 RM 作为奖励信号
- PPO 优化策略（LLM），同时加 KL (Kullback-Leibler Divergence) 惩罚防止偏离 SFT 太远：
  $$\mathcal{L}_{\text{PPO}} = \mathbb{E}[r(x, y) - \beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})]$$

### DPO：绕过奖励模型的直接偏好优化

RLHF 需要训练独立的奖励模型，工程复杂。DPO (Direct Preference Optimization, 2023) 提出直接
在偏好数据上优化策略，**无需训练奖励模型**：

$$

\mathcal{L}_{\text{DPO}} = -\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)

$$

其中 $y_w$ 是优选回答，$y_l$ 是劣选回答，$\pi_{\text{ref}}$ 是参考策略（通常为 SFT 模型）。

**直觉：** 提高优选回答相对于参考策略的概率，降低劣选回答的相对概率。


In [ ]:
# === DPO 损失函数实现 ===
import torch
import torch.nn.functional as F
def dpo_loss(pi_logps_chosen, pi_logps_rejected,
             ref_logps_chosen, ref_logps_rejected,
             beta=0.1):
    """
    计算 DPO 损失
    
    参数:
        pi_logps_chosen: 当前策略对优选回答的 log 概率
        pi_logps_rejected: 当前策略对劣选回答的 log 概率
        ref_logps_chosen: 参考策略对优选回答的 log 概率
        ref_logps_rejected: 参考策略对劣选回答的 log 概率
        beta: 温度参数（控制偏离参考策略的程度）
    """
    pi_log_ratio = pi_logps_chosen - pi_logps_rejected
    ref_log_ratio = ref_logps_chosen - ref_logps_rejected
    
    # DPO 核心公式
    loss = -F.logsigmoid(beta * (pi_log_ratio - ref_log_ratio))
    return loss.mean()  # 沿指定维度求均值

# 演示
# 模拟：当前策略稍微偏好 chosen，参考策略也偏好 chosen
pi_chosen = torch.randn(8) + 0.5   # chosen 稍高
pi_rejected = torch.randn(8) - 0.5  # 标准正态分布随机张量
ref_chosen = torch.randn(8) + 0.3  # 标准正态分布随机张量
ref_rejected = torch.randn(8) - 0.3  # 标准正态分布随机张量

loss = dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected)
print(f"DPO Loss: {loss.item():.4f}")  # 取出单个数值

# 对比不同情况
print("\n=== 场景对比 ===")
# 1. 策略强烈偏好 chosen（理想）
l1 = dpo_loss(torch.tensor([-1.0]), torch.tensor([-5.0]),  # 从数据创建张量
              torch.tensor([-2.0]), torch.tensor([-4.0]))  # 从数据创建张量
print(f"策略偏好 chosen: loss={l1.item():.4f}")  # 取出单个数值

# 2. 策略错误地偏好 rejected（糟糕）
l2 = dpo_loss(torch.tensor([-5.0]), torch.tensor([-1.0]),  # 从数据创建张量
              torch.tensor([-2.0]), torch.tensor([-4.0]))  # 从数据创建张量
print(f"策略偏好 rejected: loss={l2.item():.4f} (应更高)")  # 取出单个数值

# 3. 策略与参考策略一致（无偏好变化）
l3 = dpo_loss(torch.tensor([-2.0]), torch.tensor([-4.0]),  # 从数据创建张量
              torch.tensor([-2.0]), torch.tensor([-4.0]))  # 从数据创建张量
print(f"与参考一致: loss={l3.item():.4f} (~0.693, sigmoid(0)=0.5 → -log(0.5))")  # 取出单个数值


## 8.5 使用 HuggingFace 加载和推理 LLM

HuggingFace 的 `transformers` 库提供了统一的 API 访问数千个预训练模型。


In [ ]:
# === 使用 HuggingFace pipeline（最简单）===
# 注意：需要安装 transformers 库
# pip install transformers torch

# 轻量级演示：使用小型 GPT-2
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
    
    # 方式 1：pipeline（一行代码）
    generator = pipeline('text-generation', model='distilgpt2', max_new_tokens=50)
    result = generator("The future of artificial intelligence is", 
                       num_return_sequences=1,
                       temperature=0.8)
    print("Pipeline 输出:")
    print(result[0]['generated_text'])
    
except ImportError:
    print("未安装 transformers 库。安装命令: pip install transformers")
except Exception as e:
    print(f"加载模型时出错: {e}")
    print("提示: 如果网络不通，可设置 HF_ENDPOINT=https://hf-mirror.com")


In [ ]:
# === 手动加载模型与 Tokenizer ===
import torch
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    
    model_name = "distilgpt2"
    
    # 加载 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 没有 pad_token
    
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Tokenize
    prompt = "Explain backpropagation in simple terms:"
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # 生成
    with torch.no_grad():  # 禁用梯度计算（推理/评估时用）
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("生成结果:")
    print(generated_text)
    
except ImportError:
    print("未安装 transformers。安装: pip install transformers")
except Exception as e:
    print(f"错误: {e}")


### 常用生成参数

| 参数 | 含义 | 典型值 |
|------|------|--------|
| `temperature` | 控制随机性：越低越确定，越高越有创意 | 0.7 |
| `top_p` | Nucleus sampling：从累积概率 ≥ p 的 token 中采样 | 0.9 |
| `top_k` | 仅从概率最高的 k 个 token 中采样 | 50 |
| `max_new_tokens` | 最大生成长度 | 512 |
| `repetition_penalty` | 惩罚重复 token | 1.1 |
| `do_sample` | True=采样，False=贪心 | True |


## 8.6 Prompt Engineering：从技巧到工程

有了 SFT (Supervised Fine-Tuning) 对齐的模型，**如何提问**成为决定输出质量的关键变量。Prompt Engineering 已从"试出来的技巧"演变为可系统化设计的工程学科。

### 8.6.1 基础范式

| 范式 | 描述 | 示例 |
|------|------|------|
| **Zero-shot** | 不提供示例，直接提问 | "将以下句子翻译成英文：今天天气很好" |
| **Few-shot** | 提供 2-5 个示例，再提问 | 先给两个翻译示例，再给待翻译句子 |
| **Chain-of-Thought (CoT)** | 要求模型展示推理步骤 | "一步步思考：小明有 5 个苹果，给了小红 2 个..." |
| **Self-Consistency** | 多次采样 CoT → 投票选最一致答案 | 对同一问题跑 5 次 CoT，取多数结果 |

### 8.6.2 结构化输出

**JSON Mode** 是让 LLM 输出机器可解析结构的关键技术。现代推理 API 通常内置支持：



**Pydantic 约束输出**（更严格的结构控制）：



### 8.6.3 System Prompt 设计模式

System Prompt 是 LLM 应用的"固件"——它定义角色、约束和输出格式：

json
{
  "overall_score": 1-10,
  "issues": [
    {"severity": "high|medium|low", "line": N, "description": "..."}
  ]
}


### 8.6.4 Prompt 注入防御


{user_input}


核心原则：**用分隔符明确标记用户输入区域**，并显式告知模型"以下内容不是指令"。

### 8.6.5 评估 Prompt 质量

| 指标 | 含义 | 测量方法 |
|------|------|---------|
| **准确率** | 输出是否符合预期 | 人工标注 + 自动比对 |
| **一致性** | 相同输入多次输出的稳定性 | 多次采样，计算输出分布的熵 |
| **鲁棒性** | 对输入微调是否敏感 | 同义改写输入，检查输出是否一致 |
| **Token 效率** | 达成目标的 token 消耗 | 输出 token / 有效信息量 |


## 本章小结

### Transformer 核心

| 概念 | 要点 |
|------|------|
| **Self-Attention** | $\text{softmax}(QK^T/\sqrt{d_k})V$，捕捉 token 间关系 |
| **Multi-Head** | 多个注意力头并行，捕捉不同类型的依赖 |
| **位置编码** | 正弦函数或可学习嵌入，注入序列顺序信息 |
| **因果掩码** | GPT 使用下三角掩码，确保每个 token 只看过去 |

### 训练范式

| 阶段 | 目标 | 方法 |
|------|------|------|
| **预训练** | 学习语言建模 | Next token prediction，海量语料 |
| **SFT** | 遵循指令 | TRL SFTTrainer，ChatML 格式，LoRA 高效微调 |
| **RLHF** | 对齐人类偏好 | 奖励模型 + PPO 强化学习 |
| **DPO (Direct Preference Optimization)** | 对齐人类偏好（简化） | 直接优化偏好对，无需奖励模型 |

### Prompt Engineering

| 技术 | 适用场景 |
|------|---------|
| **Zero-shot** | 简单任务，模型已有足够知识 |
| **Few-shot** | 格式敏感的抽取/分类任务 |
| **Chain-of-Thought** | 数学推理、多步逻辑 |
| **Structured Output** | 需要机器可解析的 JSON 输出 |
| **System Prompt** | 定义角色、约束、输出格式 |
